# BNNR — ResNet18 / Imagewoof augmentation benchmark (Colab)

Reproducible comparison of four augmentation strategies on a **fine-grained, low-data, from-scratch** task — the regime where augmentation actually moves the needle, so the deltas are large and significant.

| Condition | Augmentation |
|-----------|--------------|
| `no_aug` | RandomResizedCrop + flip only |
| `randaugment` | + torchvision RandAugment |
| `trivialaugment` | + torchvision TrivialAugmentWide |
| `bnnr_branch_search` | Full BNNR loop (ICD + AICD + ChurchNoise) |

**Dataset:** Imagewoof2-160 (10 dog breeds, real ImageNet images) — auto-downloaded. **Model:** ResNet18, random init. **Regime:** balanced 100 images/class train, full val as fixed test, **from scratch**. No cross-validation — 5 seeds for training-variance error bars.

**5 seeds × 4 conditions = 20 runs.** ETA on a **free T4: ~1.5–2h**. Resume-safe: results stream to Google Drive after every (condition, seed); if the session dies, just re-run — completed runs are skipped.

**What lands on your Drive:**
- `bnnr_benchmarks/results_imagewoof.json` — aggregated metrics
- `bnnr_benchmarks/runs_imagewoof/<run_dir>/xai/*.png` — OptiCAM overlays (great for the 'where is the model looking' visuals)
- `bnnr_benchmarks/runs_imagewoof_archive.zip` — full archive (last cell)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/bnnr_benchmarks'
!mkdir -p {DRIVE_BASE}
print(f'Drive output directory: {DRIVE_BASE}')

## 2. Clone repo and install BNNR

In [ ]:
%cd /content
![ -d bnnr ] || git clone --depth=1 https://github.com/bnnr-team/bnnr.git
%cd /content/bnnr
!git pull --rebase
!pip install -e . -q
print('BNNR installed from:', !pwd)

## 3. GPU check + ETA

Confirm Colab gave you a GPU. A **free T4 is enough** (~1.5–2h). If you got a CPU runtime, go to **Runtime → Change runtime type → T4 GPU** and re-run from the top.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f'GPU: {name}')
    eta = {'T4': '~1.5-2h', 'A100': '~20 min', 'V100': '~45 min', 'L4': '~45 min'}
    for k, v in eta.items():
        if k in name:
            print(f'ETA: {v} for full 5-seed run')
            break
else:
    print('WARNING: no GPU. Full run takes many hours on CPU. Switch runtime.')

## 4. Dry-run preview

Prints the planned matrix without running anything.

In [ ]:
!PYTHONPATH=src python benchmarks/run_imagewoof.py \
  --seeds 42,43,44,45,46 \
  --device cuda \
  --drive-base-dir {DRIVE_BASE} \
  --dry-run

## 5. Smoke test (~1–2 minutes)

1 epoch, 5/class train, tiny val, img-size 64. Verifies the full pipeline (download → train → XAI export → JSON save) on real GPU before the long run.

Smoke results write to `{DRIVE_BASE}/results_imagewoof.json`. Cell 6 wipes them before the full run.

In [ ]:
!PYTHONPATH=src python benchmarks/run_imagewoof.py \
  --device cuda \
  --drive-base-dir {DRIVE_BASE} \
  --smoke

## 6. (Optional) Clean smoke results before full run

If you ran the smoke in cell 5, this wipes the smoke entries (epochs=1) so the full run starts clean. Keeps only full-run rows (epochs=25).

In [ ]:
import json, os
results_path = f'{DRIVE_BASE}/results_imagewoof.json'
if os.path.exists(results_path):
    with open(results_path) as f: data = json.load(f)
    before = len(data.get('runs', []))
    data['runs'] = [r for r in data.get('runs', []) if r.get('m_epochs') == 25 and 'val_metric' in r]
    after = len(data['runs'])
    with open(results_path, 'w') as f: json.dump(data, f, indent=2)
    print(f'Filtered {before} -> {after} runs (kept only m_epochs=25 with val_metric)')
else:
    print('No results file yet — nothing to clean.')

## 7. Full benchmark — 5 seeds × 4 conditions

**The long run.** Resume-safe: if Colab kills the session, just re-run this cell — already-completed (condition, seed) pairs are skipped. Keep the tab active (Colab idles out after ~90 min).

In [ ]:
!PYTHONPATH=src python benchmarks/run_imagewoof.py \
  --seeds 42,43,44,45,46 \
  --device cuda \
  --drive-base-dir {DRIVE_BASE}

## 8. Summarize → markdown table

In [ ]:
!PYTHONPATH=src python benchmarks/summarize.py \
  --results {DRIVE_BASE}/results_imagewoof.json \
  --markdown

## 9. Backup `runs_imagewoof/` as ZIP on Drive

Contains all OptiCAM overlay PNGs + per-run logs. A single ZIP is easier to download/share.

In [ ]:
import shutil
zip_path = shutil.make_archive(
    f'{DRIVE_BASE}/runs_imagewoof_archive',
    'zip',
    f'{DRIVE_BASE}/runs_imagewoof',
)
print(f'Archive created: {zip_path}')
!ls -lh {DRIVE_BASE}/

## Done

On your Drive:
- `results_imagewoof.json` — paste the table into `benchmarks/README.md` after review
- `runs_imagewoof/` — per-run logs + OptiCAM overlays (use for X/LinkedIn 'where the model looks' visuals)
- `runs_imagewoof_archive.zip` — full backup

**Next steps:**
1. Sanity-check `results_imagewoof.json` (no errors, all 20 runs present)
2. Paste the markdown table from cell 8 into `benchmarks/README.md` § Results
3. Pick the best OptiCAM overlay comparison from `runs_imagewoof/*/xai/` for the README hero